# Degradation Ablation — 5×5 Cross-Test

**Goal**: Show that 3DGS rendering-induced degradation is fundamentally different
from the classic "bicubic + Gaussian noise" assumption in standard depth SR.

## Experiment Design

**5 LR degradation variants** (all generated from the same HR depth):
1. **bicubic** — pure spatial downsampling (generic SR)
2. **bicubic+noise** — downsampling + Gaussian noise (classical depth SR)
3. **bicubic+q8** — downsampling + 8-bit quantization (pure bit-depth)
4. **render** — 3DGS render downscale (spatial component of our setting)
5. **render+q8** — full setting used in our pipeline

**5×5 cross-test matrix**: train on variant X, test on variant Y.
Strong diagonal + weak off-diagonal → each degradation has a unique pattern
that cannot be substituted. This supports claim (a).

**Training cost**:
- Quick: 50 ep × 5 runs × 250s ≈ **17 hours**
- Full: 150 ep × 5 runs × 250s ≈ **52 hours**

Cross-test itself (after training) takes only a few minutes.


## 0. Setup


In [ ]:
# Shared setup
from pathlib import Path
import os, sys, time, json, shutil, subprocess
import numpy as np
import cv2
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "FaceLift" else Path.cwd().resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

DATASET_DIR = PROJECT_ROOT / "data" / "dataset"
CKPT_ROOT   = PROJECT_ROOT / "checkpoints"
DEG_ROOT    = PROJECT_ROOT / "data" / "degradation_variants"   # LR variants go here
DEG_ROOT.mkdir(parents=True, exist_ok=True)

# Reuse canonical trainer components
from train_depth_upres import DepthUpResDataset, DepthUpResUNet, depth_loss, _masked_l1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Variants we're going to compare
VARIANTS = ["bicubic", "bicubic_noise", "bicubic_q8", "render", "render_q8"]
print(f"Variants: {VARIANTS}")


## 1. Generate 5 LR variants from the same HR depth

Each sample gets 5 LR versions, one per degradation model. Written to
`data/degradation_variants/{train,val}/{variant_name}/*.png`.

`render` / `render_q8` reuse existing data from
`data/dataset/.../depth_lr_16bit` and `depth_lr_8bit`
(already render-based). The other 3 are regenerated from HR.


In [ ]:
# 1.1 Pure bicubic downsample (generic SR degradation)
# Neither PIL 'I;16' nor cv2 uint16 support BICUBIC on Windows.
# Workaround: cast to float32, resize, clip+cast back.
def gen_bicubic(hr_uint16, target_size=256):
    """Input: HR uint16 (1024,1024). Output: LR uint16 (256,256)."""
    hr_f = hr_uint16.astype(np.float32)
    lr_f = cv2.resize(hr_f, (target_size, target_size),
                      interpolation=cv2.INTER_CUBIC)
    return np.clip(lr_f, 0, 65535).astype(np.uint16)

# 1.2 Bicubic + Gaussian noise (classical depth SR degradation)
def gen_bicubic_noise(hr_uint16, target_size=256, sigma_pct=2.0):
    """Bicubic + additive Gaussian noise (~sigma_pct% of depth range)."""
    lr = gen_bicubic(hr_uint16, target_size).astype(np.float32)
    sigma = 65535 * sigma_pct / 100.0
    lr = lr + np.random.normal(0, sigma, lr.shape)
    return np.clip(lr, 0, 65535).astype(np.uint16)

# 1.3 Bicubic + 8-bit quantization (pure bit-depth compression)
def gen_bicubic_q8(hr_uint16, target_size=256):
    lr16 = gen_bicubic(hr_uint16, target_size).astype(np.float32) / 65535.0
    lr8  = np.clip(lr16 * 255 + 0.5, 0, 255).astype(np.uint8)
    # Scale back to uint16 so all variants share the same on-disk bit-depth
    # (simplifies the loader); 8-bit banding is already baked in.
    lr16_back = (lr8.astype(np.float32) / 255.0 * 65535).astype(np.uint16)
    return lr16_back

# 1.4 / 1.5 render / render_q8 — reuse existing data
# TODO: symlink / copy stage pointing to existing
#       dataset/{split}/depth_lr_{16bit,8bit}/ as render variants.


# Generate all variants for train + val
def generate_variant(variant_name, split):
    """Write LR PNGs for one variant + one split."""
    out_dir = DEG_ROOT / split / variant_name
    out_dir.mkdir(parents=True, exist_ok=True)
    hr_dir = DATASET_DIR / split / "depth"
    stems = sorted(p.stem for p in hr_dir.glob("*.png"))
    
    # Fast skip-check
    if len(list(out_dir.glob("*.png"))) == len(stems):
        print(f"  [SKIP {variant_name}/{split}] already done: {len(stems)} files")
        return
    
    for stem in stems:
        out_path = out_dir / f"{stem}.png"
        if out_path.exists(): continue
        hr = np.array(Image.open(hr_dir / f"{stem}.png"))  # uint16
        
        if variant_name == "bicubic":
            lr = gen_bicubic(hr)
        elif variant_name == "bicubic_noise":
            lr = gen_bicubic_noise(hr)
        elif variant_name == "bicubic_q8":
            lr = gen_bicubic_q8(hr)
        elif variant_name == "render":
            # Reuse dataset/{split}/depth_lr_16bit
            src = DATASET_DIR / split / "depth_lr_16bit" / f"{stem}.png"
            shutil.copy(src, out_path)
            continue
        elif variant_name == "render_q8":
            # Reuse dataset/{split}/depth_lr_8bit, converted to uint16 for uniformity
            src = DATASET_DIR / split / "depth_lr_8bit" / f"{stem}.png"
            lr8 = np.array(Image.open(src))  # uint8
            lr = (lr8.astype(np.float32) / 255.0 * 65535).astype(np.uint16)
        else:
            raise ValueError(variant_name)
        
        Image.fromarray(lr, mode="I;16").save(out_path)
    
    print(f"  [{variant_name}/{split}] wrote {len(stems)} files to {out_dir}")


np.random.seed(42)
for split in ("train", "val"):
    print(f"--- {split} ---")
    for v in VARIANTS:
        generate_variant(v, split)

print("\nAll variants generated.")


## 2. Train 5 UNets, one per variant

Same `DepthUpResUNet(base_ch=32)`, same `depth_loss`, same Adam + cosine.
Only the LR variant changes.

**Epoch budget**:
- Quick: `EPOCHS = 50` → total ~17h (overnight)
- Full: `EPOCHS = 150` → total ~52h (3 days)

Run quick first. If the cross-test signal is clear, that's enough.


In [ ]:
# 2. Train 5 UNets with a custom dataset pointing at the variant LR

class VariantDataset(torch.utils.data.Dataset):
    """Depth SR dataset where LR reads from data/degradation_variants/{split}/{variant}/"""
    def __init__(self, split, variant):
        self.hr_dir   = DATASET_DIR / split / "depth"
        self.lr_dir   = DEG_ROOT    / split / variant
        self.mask_dir = DATASET_DIR / split / "mask"
        self.has_mask = self.mask_dir.exists() and any(self.mask_dir.glob("*.png"))
        self.samples  = sorted(p.stem for p in self.hr_dir.glob("*.png"))
    
    def __len__(self): return len(self.samples)
    
    def __getitem__(self, idx):
        name = self.samples[idx]
        hr_pil = Image.open(self.hr_dir / f"{name}.png")
        hr = np.asarray(hr_pil)
        hr_f = (hr.astype(np.float32) / 65535.0) if hr.max() > 255 else (hr.astype(np.float32) / 255.0)
        
        lr_pil = Image.open(self.lr_dir / f"{name}.png")
        lr = np.asarray(lr_pil)
        lr_f = (lr.astype(np.float32) / 65535.0) if lr.max() > 255 else (lr.astype(np.float32) / 255.0)
        
        if self.has_mask:
            mp = self.mask_dir / f"{name}.png"
            if mp.exists():
                with Image.open(mp) as img:
                    m = np.array(img.convert("L"))
                m_f = (m > 127).astype(np.float32)
            else:
                m_f = np.ones_like(hr_f, dtype=np.float32)
        else:
            m_f = np.ones_like(hr_f, dtype=np.float32)
        
        hr_t = torch.from_numpy(hr_f).unsqueeze(0)
        lr_t = torch.from_numpy(lr_f).unsqueeze(0)
        m_t  = torch.from_numpy(m_f).unsqueeze(0)
        return lr_t, hr_t, m_t


from torch.utils.data import DataLoader

EPOCHS = 50              # quick: 50 · complete: 150
BATCH  = 1
GRAD_ACC = 8
LR_ = 1e-4

for variant in VARIANTS:
    ckpt_dir = CKPT_ROOT / f"degradation_{variant}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    if (ckpt_dir / "best.pt").exists():
        print(f"[SKIP] {variant} already trained")
        continue
    
    print(f"\n=== Training variant: {variant} ===")
    train_ds = VariantDataset("train", variant)
    val_ds   = VariantDataset("val",   variant)
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)
    
    model = DepthUpResUNet(base_ch=32).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR_, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.cuda.amp.GradScaler()
    best_val = float("inf"); history = []
    
    for epoch in range(1, EPOCHS + 1):
        model.train(); t0 = time.time(); tr = 0.0
        opt.zero_grad()
        for step, (lr_in, hr, m) in enumerate(train_loader):
            lr_in, hr, m = [t.to(device, non_blocking=True) for t in (lr_in, hr, m)]
            with torch.cuda.amp.autocast():
                pred = model(lr_in)
                loss = depth_loss(pred, hr, mask=m) / GRAD_ACC
            scaler.scale(loss).backward()
            if (step + 1) % GRAD_ACC == 0:
                scaler.step(opt); scaler.update(); opt.zero_grad()
            tr += loss.item() * GRAD_ACC
        tr /= len(train_ds)
        
        model.eval(); vl = 0.0; v_l1 = 0.0
        with torch.no_grad():
            for lr_in, hr, m in val_loader:
                lr_in, hr, m = [t.to(device, non_blocking=True) for t in (lr_in, hr, m)]
                pred = model(lr_in)
                vl   += depth_loss(pred, hr, mask=m).item()
                v_l1 += _masked_l1(pred.float(), hr, m).item()
        vl   /= len(val_ds); v_l1 /= len(val_ds)
        sched.step()
        
        history.append({"epoch": epoch, "train": tr, "val": vl, "val_l1": v_l1,
                        "lr": opt.param_groups[0]["lr"], "time_s": time.time() - t0})
        print(f"  [{variant} {epoch:03d}/{EPOCHS}] train={tr:.5f} val={vl:.5f} "
              f"val_l1={v_l1:.5f} time={time.time()-t0:.0f}s")
        
        if vl < best_val:
            best_val = vl
            torch.save({"model": model.state_dict(), "variant": variant,
                        "epoch": epoch, "val_loss": vl,
                        "args": {"base_ch": 32}}, ckpt_dir / "best.pt")
        (ckpt_dir / "train_log.json").write_text(json.dumps(history, indent=2))
    
    print(f"Done {variant}. Best val: {best_val:.5f}")


## 3. 5×5 Cross-Test Matrix

For each trained model (5 total), evaluate on every variant's val set (5 total).
Produces a 5×5 L1 matrix.

**Diagonal**: model trained and tested on the same variant → "home turf"
**Off-diagonal**: cross-test → measures generalization

If **diagonal is significantly better than off-diagonal**, each degradation has
a unique pattern that models cannot transfer across. Supports claim (a).


In [ ]:
# 3. Cross-test: all models × all variants
results = np.zeros((len(VARIANTS), len(VARIANTS)))   # [train_variant, test_variant]
psnr_mat = np.zeros_like(results)

for i, train_v in enumerate(VARIANTS):
    ckpt_path = CKPT_ROOT / f"degradation_{train_v}" / "best.pt"
    if not ckpt_path.exists():
        print(f"  missing {ckpt_path}, skipping row")
        continue
    model = DepthUpResUNet(base_ch=32).to(device).eval()
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=False)["model"])
    
    for j, test_v in enumerate(VARIANTS):
        val_ds = VariantDataset("val", test_v)
        val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)
        total_l1 = 0.0; total_mse = 0.0
        with torch.no_grad():
            for lr_in, hr, m in val_loader:
                lr_in, hr, m = [t.to(device) for t in (lr_in, hr, m)]
                pred = model(lr_in).clamp(0, 1)
                l1 = _masked_l1(pred, hr, m).item()
                mse = ((pred - hr)**2 * m).sum() / m.sum().clamp(min=1e-6)
                total_l1  += l1
                total_mse += mse.item()
        l1   = total_l1  / len(val_ds)
        mse  = total_mse / len(val_ds)
        psnr = 20 * np.log10(1.0) - 10 * np.log10(mse + 1e-12)
        results[i, j]  = l1
        psnr_mat[i, j] = psnr
        print(f"  {train_v} → {test_v}:  L1 = {l1:.5f}  PSNR = {psnr:.2f} dB")

# Save as CSV
df_l1   = pd.DataFrame(results,  index=[f"train_{v}" for v in VARIANTS],
                                  columns=[f"test_{v}"  for v in VARIANTS])
df_psnr = pd.DataFrame(psnr_mat, index=[f"train_{v}" for v in VARIANTS],
                                  columns=[f"test_{v}"  for v in VARIANTS])
Path("eval").mkdir(exist_ok=True)
df_l1.to_csv("eval/degradation_crosstest_L1.csv")
df_psnr.to_csv("eval/degradation_crosstest_PSNR.csv")
print("\nSaved to eval/degradation_crosstest_{L1,PSNR}.csv")
display(df_l1.round(5))
display(df_psnr.round(2))


## 4. Visualize cross-test heatmap

Diagonal should be visibly deeper (lower L1 = better).
If the matrix is roughly uniform, degradations are interchangeable
and claim (a) is weak.


In [ ]:
# 4. Cross-test heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# L1 (lower = better, reversed colormap)
im0 = axes[0].imshow(results, cmap="viridis_r", aspect="auto")
axes[0].set_xticks(range(len(VARIANTS))); axes[0].set_xticklabels(VARIANTS, rotation=30, ha="right")
axes[0].set_yticks(range(len(VARIANTS))); axes[0].set_yticklabels(VARIANTS)
axes[0].set_xlabel("Test variant"); axes[0].set_ylabel("Train variant")
axes[0].set_title("L1 (lower = better)")
for i in range(len(VARIANTS)):
    for j in range(len(VARIANTS)):
        axes[0].text(j, i, f"{results[i,j]:.4f}", ha="center", va="center",
                     color="white" if results[i,j] > results.mean() else "black", fontsize=9)
plt.colorbar(im0, ax=axes[0])

# PSNR (higher = better)
im1 = axes[1].imshow(psnr_mat, cmap="viridis", aspect="auto")
axes[1].set_xticks(range(len(VARIANTS))); axes[1].set_xticklabels(VARIANTS, rotation=30, ha="right")
axes[1].set_yticks(range(len(VARIANTS))); axes[1].set_yticklabels(VARIANTS)
axes[1].set_xlabel("Test variant"); axes[1].set_ylabel("Train variant")
axes[1].set_title("PSNR (higher = better, dB)")
for i in range(len(VARIANTS)):
    for j in range(len(VARIANTS)):
        axes[1].text(j, i, f"{psnr_mat[i,j]:.1f}", ha="center", va="center",
                     color="white" if psnr_mat[i,j] < psnr_mat.mean() else "black", fontsize=9)
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.savefig("eval/degradation_crosstest_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved heatmap to eval/degradation_crosstest_heatmap.png")

# --- Quantitative check: is diagonal significantly better than off-diagonal? ---
diag = np.diag(results)
off  = results[~np.eye(len(VARIANTS), dtype=bool)]
print(f"\nL1 diagonal mean    = {diag.mean():.5f}   (best on own train distribution)")
print(f"L1 off-diagonal mean = {off.mean():.5f}")
print(f"L1 gap (off/diag − 1) = {off.mean()/diag.mean() - 1:.2%}  → larger gap = stronger claim (a)")
